# Assorted Medicine Dataset of Bangladesh
### Data Cleaning & Master Dataset Preparation

**Overview**

This notebook focuses on exploring, cleaning, standardizing, and consolidating medicine-related data from the [Assorted Medicine Dataset of Bangladesh](https://www.kaggle.com/datasets/ahmedshahriarsakib/assorted-medicine-dataset-of-bangladesh) to build a reliable and structured master dataset for the OushudhCheck project.

In [19]:
!pip install -q kagglehub

In [34]:
import kagglehub
import numpy as np
import pandas as pd
import os
from pathlib import Path
import re
import html

In [21]:
DATA_DIR = kagglehub.dataset_download('ahmedshahriarsakib/assorted-medicine-dataset-of-bangladesh')
DATA_DIR = Path(DATA_DIR)
print("Dataset downloaded to:", DATA_DIR)

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Using Colab cache for faster access to the 'assorted-medicine-dataset-of-bangladesh' dataset.
Dataset downloaded to: /kaggle/input/assorted-medicine-dataset-of-bangladesh
/kaggle/input/assorted-medicine-dataset-of-bangladesh/manufacturer.csv
/kaggle/input/assorted-medicine-dataset-of-bangladesh/drug class.csv
/kaggle/input/assorted-medicine-dataset-of-bangladesh/medicine.csv
/kaggle/input/assorted-medicine-dataset-of-bangladesh/generic.csv
/kaggle/input/assorted-medicine-dataset-of-bangladesh/dosage form.csv
/kaggle/input/assorted-medicine-dataset-of-bangladesh/indication.csv


In [22]:
# Making One Dataset

# 1. Load all 6 CSVs
medicine     = pd.read_csv(DATA_DIR / "medicine.csv", low_memory=False)
generic      = pd.read_csv(DATA_DIR / "generic.csv", low_memory=False)
manufacturer = pd.read_csv(DATA_DIR / "manufacturer.csv", low_memory=False)
dosage_form  = pd.read_csv(DATA_DIR / "dosage form.csv", low_memory=False)
drug_class   = pd.read_csv(DATA_DIR / "drug class.csv", low_memory=False)
indication   = pd.read_csv(DATA_DIR / "indication.csv", low_memory=False)

# Display dataset dimensions
datasets = {
    "Medicine": medicine,
    "Generic": generic,
    "Manufacturer": manufacturer,
    "Dosage Form": dosage_form,
    "Drug Class": drug_class,
    "Indication": indication
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Medicine: (21714, 10)
Generic: (1711, 22)
Manufacturer: (240, 5)
Dosage Form: (113, 4)
Drug Class: (453, 4)
Indication: (2043, 4)


In [31]:
def clean_key(series):
    return series.astype(str).str.strip().str.lower()

medicine["_generic_key"]        = clean_key(medicine["generic"])
generic["_generic_key"]         = clean_key(generic["generic name"])
medicine["_manufacturer_key"]   = clean_key(medicine["manufacturer"])
manufacturer["_manufacturer_key"] = clean_key(manufacturer["manufacturer name"])
medicine["_dosage_key"]         = clean_key(medicine["dosage form"])
dosage_form["_dosage_key"]      = clean_key(dosage_form["dosage form name"])

# Deduplicate lookup tables
generic_clean      = generic.drop_duplicates(subset=["_generic_key"])
manufacturer_clean = manufacturer.drop_duplicates(subset=["_manufacturer_key"])
dosage_clean       = dosage_form.drop_duplicates(subset=["_dosage_key"])

# Merge medicine -> generic
master = medicine.merge(
    generic_clean,
    on="_generic_key",
    how="left",
    suffixes=("_brand", "_generic")
)

# Merge remaining lookup tables
master = master.merge(
    manufacturer_clean.add_prefix("mfg_"),
    left_on="_manufacturer_key",
    right_on="mfg__manufacturer_key",
    how="left"
)

master = master.merge(
    dosage_clean.add_prefix("df_"),
    left_on="_dosage_key",
    right_on="df__dosage_key",
    how="left"
)

if "drug class" in master.columns and "drug class name" in drug_class.columns:
    master["_class_key"] = clean_key(master["drug class"])
    drug_class["_class_key"] = clean_key(drug_class["drug class name"])
    master = master.merge(
        drug_class.drop_duplicates(subset=["_class_key"]).add_prefix("class_"),
        left_on="_class_key",
        right_on="class__class_key",
        how="left"
    )

if "indication" in master.columns and "indication name" in indication.columns:
    master["_indication_key"] = clean_key(master["indication"])
    indication["_indication_key"] = clean_key(indication["indication name"])
    master = master.merge(
        indication.drop_duplicates(subset=["_indication_key"]).add_prefix("ind_"),
        left_on="_indication_key",
        right_on="ind__indication_key",
        how="left"
    )

# Drop temporary join-key columns
temp_cols = [c for c in master.columns if "_key" in c]
master.drop(columns=temp_cols, inplace=True)

# Verify and save
print(f"Original medicine rows: {len(medicine)}")
print(f"Master CSV shape:       {master.shape}")

output_path = "/content/master_medicine_bd.csv"
master.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Original medicine rows: 21714
Master CSV shape:       (21714, 49)
Saved to /content/master_medicine_bd.csv


In [32]:
df = pd.read_csv("/content/master_medicine_bd.csv")

print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()


Shape: (21714, 49)
Columns: ['brand id', 'brand name', 'type', 'slug_brand', 'dosage form', 'generic', 'strength', 'manufacturer', 'package container', 'Package Size', 'generic id', 'generic name', 'slug_generic', 'monograph link', 'drug class', 'indication', 'indication description', 'therapeutic class description', 'pharmacology description', 'dosage description', 'administration description', 'interaction description', 'contraindications description', 'side effects description', 'pregnancy and lactation description', 'precautions description', 'pediatric usage description', 'overdose effects description', 'duration of treatment description', 'reconstitution description', 'storage conditions description', 'descriptions count', 'mfg_manufacturer id', 'mfg_manufacturer name', 'mfg_slug', 'mfg_generics count', 'mfg_brand names count', 'df_dosage form id', 'df_dosage form name', 'df_slug', 'df_brand names count', 'class_drug class id', 'class_drug class name', 'class_slug', 'class_generi

,brand id,brand name,type,slug_brand,dosage form,generic,strength,manufacturer,package container,Package Size,...,df_slug,df_brand names count,class_drug class id,class_drug class name,class_slug,class_generics count,ind_indication id,ind_indication name,ind_slug,ind_generics count
0,4077,A-Cold,allopathic,a-coldsyrup4-mg5-ml,Syrup,Bromhexine Hydrochloride,4 mg/5 ml,ACME Laboratories Ltd.,100 ml bottle: ৳ 40.12,NaN,...,syrup-3,1010,488.0,Cough expectorants & mucolytics,cough-expectorants-mucolytics-488,6.0,2209.0,Bronchitis,bronchitis-2209,32.0
1,4006,A-Cof,allopathic,a-cofsyrup10-mg30-mg125-mg5-ml,Syrup,Dextromethorphan + Pseudoephedrine + Triprolidine,(10 mg+30 mg+1.25 mg)/5 ml,ACME Laboratories Ltd.,100 ml bottle: ৳ 100.00,NaN,...,syrup-3,1010,487.0,Combined cough suppressants,combined-cough-suppressants-487,5.0,2990.0,Non-productive cough,non-productive-cough-2990,4.0
2,6174,A-Clox,allopathic,a-cloxinjection500-mgvial,Injection,Cloxacillin Sodium,500 mg/vial,ACME Laboratories Ltd.,"500 mg vial: ৳ 28.43,(5's pack: ৳ 142.15),",(5's pack: ৳ 142.15),...,injection-12,234,71.0,Penicillinase-resistant penicillins,penicillinase-resistant-penicillins-71,2.0,3540.0,UTI caused by Staphylococcus Aureus,uti-caused-by-staphylococcus-aureus-3540,1.0
3,6173,A-Clox,allopathic,a-cloxinjection250-mgvial,Injection,Cloxacillin Sodium,250 mg/vial,ACME Laboratories Ltd.,"250 mg vial: ৳ 20.00,(5's pack: ৳ 100.00),",(5's pack: ৳ 100.00),...,injection-12,234,71.0,Penicillinase-resistant penicillins,penicillinase-resistant-penicillins-71,2.0,3540.0,UTI caused by Staphylococcus Aureus,uti-caused-by-staphylococcus-aureus-3540,1.0
4,6172,A-Clox,allopathic,a-cloxpowder-for-suspension125-mg5-ml,Powder for Suspension,Cloxacillin Sodium,125 mg/5 ml,ACME Laboratories Ltd.,100 ml bottle: ৳ 45.00,NaN,...,powder-for-suspension-75,1011,71.0,Penicillinase-resistant penicillins,penicillinase-resistant-penicillins-71,2.0,3540.0,UTI caused by Staphylococcus Aureus,uti-caused-by-staphylococcus-aureus-3540,1.0


In [33]:
df['Package Size'].head(10) #  'Package Size'
df.head(5).T

,0,1,2,3,4
brand id,4077,4006,6174,6173,6172
brand name,A-Cold,A-Cof,A-Clox,A-Clox,A-Clox
type,allopathic,allopathic,allopathic,allopathic,allopathic
slug_brand,a-coldsyrup4-mg5-ml,a-cofsyrup10-mg30-mg125-mg5-ml,a-cloxinjection500-mgvial,a-cloxinjection250-mgvial,a-cloxpowder-for-suspension125-mg5-ml
dosage form,Syrup,Syrup,Injection,Injection,Powder for Suspension
generic,Bromhexine Hydrochloride,Dextromethorphan + Pseudoephedrine + Triprolidine,Cloxacillin Sodium,Cloxacillin Sodium,Cloxacillin Sodium
strength,4 mg/5 ml,(10 mg+30 mg+1.25 mg)/5 ml,500 mg/vial,250 mg/vial,125 mg/5 ml
manufacturer,ACME Laboratories Ltd.,ACME Laboratories Ltd.,ACME Laboratories Ltd.,ACME Laboratories Ltd.,ACME Laboratories Ltd.
package container,100 ml bottle: ৳ 40.12,100 ml bottle: ৳ 100.00,"500 mg vial: ৳ 28.43,(5's pack: ৳ 142.15),","250 mg vial: ৳ 20.00,(5's pack: ৳ 100.00),",100 ml bottle: ৳ 45.00
Package Size,NaN,NaN,(5's pack: ৳ 142.15),(5's pack: ৳ 100.00),NaN


In [35]:
# Drop redundant lookup columns, slugs, and duplicate IDs/names
cols_to_drop = [
    c for c in df.columns
    if c.startswith(("mfg_", "df_", "class_", "ind_"))
    or "slug" in c.lower()
    or c in ["generic id", "generic name", "descriptions count"]
]
df = df.drop(columns=cols_to_drop, errors="ignore")

# Strip HTML tags & decode entities (&amp;, &nbsp;, etc.) in all description columns
TAG_RE = re.compile(r"<[^>]+>")
SPACE_RE = re.compile(r"\s+")

def clean_html(val):
    if pd.isna(val):
        return pd.NA
    text = html.unescape(str(val))
    text = TAG_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return text if text else pd.NA

desc_cols = [c for c in df.columns if "description" in c.lower()]
for col in desc_cols:
    df[col] = df[col].apply(clean_html)

# Extract numeric prices (BDT) from package columns
PRICE_RE = re.compile(r"৳\s*([\d,]+(?:\.\d+)?)")

def extract_price(val):
    if pd.isna(val):
        return pd.NA
    match = PRICE_RE.search(str(val))
    return float(match.group(1).replace(",", "")) if match else pd.NA

df["unit_price_bdt"] = df["package container"].apply(extract_price)
df["pack_price_bdt"] = df["Package Size"].apply(extract_price)

# Clean trailing commas/whitespace in package container
df["package container"] = df["package container"].astype("string").str.strip(" ,")

# Flag rows that contain Bengali text in descriptions (useful if you want English-only later)
BENGALI_RE = re.compile(r"[\u0980-\u09FF]")
df["has_bengali_desc"] = df["indication description"].fillna("").apply(
    lambda x: bool(BENGALI_RE.search(x))
)

# Standardize column names to snake_case & drop exact duplicate rows
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
df = df.drop_duplicates()

# Save clean dataset
output_path = "/content/master_medicine_bd_clean.csv"
df.to_csv(output_path, index=False)

print("Clean shape:", df.shape)
print("\nMissing values (%):")
print((df.isna().mean() * 100).round(1).sort_values(ascending=False))

Clean shape: (21714, 30)

Missing values (%):
duration_of_treatment_description      99.9
reconstitution_description             89.2
administration_description             73.8
pediatric_usage_description            62.3
overdose_effects_description           42.1
package_size                           35.8
pack_price_bdt                         35.8
monograph_link                         18.6
interaction_description                 9.5
indication                              7.9
precautions_description                 4.6
drug_class                              4.5
therapeutic_class_description           4.4
strength                                3.9
contraindications_description           3.3
storage_conditions_description          2.9
pharmacology_description                2.8
pregnancy_and_lactation_description     2.3
side_effects_description                0.9
unit_price_bdt                          0.4
package_container                       0.2
dosage_description            

In [38]:
threshold = 0.1
df = df.loc[:, df.isna().mean() < threshold]
df.to_csv("/content/master_medicine_bd_clean.csv", index=False)
print("Shape after dropping sparse columns:", df.shape)

Shape after dropping sparse columns: (21714, 22)


In [39]:
df.head().T

,0,1,2,3,4
brand_id,4077,4006,6174,6173,6172
brand_name,A-Cold,A-Cof,A-Clox,A-Clox,A-Clox
type,allopathic,allopathic,allopathic,allopathic,allopathic
dosage_form,Syrup,Syrup,Injection,Injection,Powder for Suspension
generic,Bromhexine Hydrochloride,Dextromethorphan + Pseudoephedrine + Triprolidine,Cloxacillin Sodium,Cloxacillin Sodium,Cloxacillin Sodium
strength,4 mg/5 ml,(10 mg+30 mg+1.25 mg)/5 ml,500 mg/vial,250 mg/vial,125 mg/5 ml
manufacturer,ACME Laboratories Ltd.,ACME Laboratories Ltd.,ACME Laboratories Ltd.,ACME Laboratories Ltd.,ACME Laboratories Ltd.
package_container,100 ml bottle: ৳ 40.12,100 ml bottle: ৳ 100.00,"500 mg vial: ৳ 28.43,(5's pack: ৳ 142.15)","250 mg vial: ৳ 20.00,(5's pack: ৳ 100.00)",100 ml bottle: ৳ 45.00
drug_class,Cough expectorants & mucolytics,Combined cough suppressants,Penicillinase-resistant penicillins,Penicillinase-resistant penicillins,Penicillinase-resistant penicillins
indication,Bronchitis,Non-productive cough,UTI caused by Staphylococcus Aureus,UTI caused by Staphylococcus Aureus,UTI caused by Staphylococcus Aureus


In [40]:
path = "/content/master_medicine_bd_clean.csv"
df = pd.read_csv(path)

df = df.drop(columns=["unit_price_bdt", "has_bengali_desc", "strength"], errors="ignore")

# Keep only the container text before ':' or '৳'
df["package_container"] = (
    df["package_container"]
    .astype("string")
    .str.replace(r"[:৳].*$", "", regex=True)
    .str.strip(" ,")
)

# Save and verify
df.to_csv(path, index=False)

print("New shape:", df.shape)
print(df["package_container"].head(10))

New shape: (21714, 19)
0    100 ml bottle
1    100 ml bottle
2      500 mg vial
3      250 mg vial
4    100 ml bottle
5       Unit Price
6       Unit Price
7       Unit Price
8       Unit Price
9       Unit Price
Name: package_container, dtype: string


In [42]:
invalid_pattern = r"^(unit price|price unavailable|not for sale.*|)$"

df["package_container"] = (
    df["package_container"]
    .astype("string")
    .str.strip()
    .replace(invalid_pattern, np.nan, regex=True)
)

# Also catch mixed-case variants
mask = df["package_container"].str.lower().str.match(invalid_pattern, na=False)
df.loc[mask, "package_container"] = np.nan

df.to_csv(path, index=False)

print("Null count:", df["package_container"].isna().sum())

Null count: 13579


In [49]:
df.head(5).T

,0,1,2,3,4
brand_id,4077,4006,6174,6173,6172
brand_name,A-Cold,A-Cof,A-Clox,A-Clox,A-Clox
type,allopathic,allopathic,allopathic,allopathic,allopathic
dosage_form,Syrup,Syrup,Injection,Injection,Powder for Suspension
generic,Bromhexine Hydrochloride,Dextromethorphan + Pseudoephedrine + Triprolidine,Cloxacillin Sodium,Cloxacillin Sodium,Cloxacillin Sodium
manufacturer,ACME Laboratories Ltd.,ACME Laboratories Ltd.,ACME Laboratories Ltd.,ACME Laboratories Ltd.,ACME Laboratories Ltd.
package_container,100 ml bottle,100 ml bottle,500 mg vial,250 mg vial,100 ml bottle
drug_class,Cough expectorants & mucolytics,Combined cough suppressants,Penicillinase-resistant penicillins,Penicillinase-resistant penicillins,Penicillinase-resistant penicillins
indication,Bronchitis,Non-productive cough,UTI caused by Staphylococcus Aureus,UTI caused by Staphylococcus Aureus,UTI caused by Staphylococcus Aureus
indication_description,Bromhexine is indicated in the treatment of re...,This preparation temporarily relieves these sy...,গ্রাম-পজিটিভ জীবাণু দ্বারা সৃষ্ট সংক্রমণের চিক...,গ্রাম-পজিটিভ জীবাণু দ্বারা সৃষ্ট সংক্রমণের চিক...,গ্রাম-পজিটিভ জীবাণু দ্বারা সৃষ্ট সংক্রমণের চিক...


In [53]:
df[df["brand_name"] == "Napa Extra"]

,brand_id,brand_name,type,dosage_form,generic,manufacturer,package_container,drug_class,indication,indication_description,therapeutic_class_description,pharmacology_description,dosage_description,interaction_description,contraindications_description,side_effects_description,pregnancy_and_lactation_description,precautions_description,storage_conditions_description
12989,10592,Napa Extra,allopathic,Tablet,Paracetamol + Caffeine,Beximco Pharmaceuticals Ltd.,<NA>,Non opioid analgesics,Toothache,The is indicated in the following condition- H...,Non opioid analgesics,This is a combination of Paracetamol and Caffe...,Adult dose : 1-2 tablets every 4-6 hours. Maxi...,May reduce serum levels with anticonvulsants (...,Paracetamol is contraindicated in patients wit...,"Side effects of paracetamol are usually mild, ...",Pregnant mothers should consult with doctors b...,Paracetamol & Caffeine should be given cautiou...,"Store in a cool and dry place, protect from li..."


In [52]:
# Find all rows where 'generic' contains the word "generic" (case-insensitive)
matches = df[df["generic"].str.contains("Paracetamol", case=False, na=False)]

print("Matching rows:", len(matches))
matches[["brand_name", "generic", "manufacturer"]]

Matching rows: 358


,brand_name,generic,manufacturer
649,Alkaparol,Paracetamol,Alkad Laboratories
1026,Ampol XR,Paracetamol,Al-Madina Pharmaceuticals Ltd.
1027,Ampol Plus,Paracetamol + Caffeine,Al-Madina Pharmaceuticals Ltd.
1097,Anapol Plus,Paracetamol + Caffeine,Pharmik Laboratories Ltd.
1098,Anapol,Paracetamol,Pharmik Laboratories Ltd.
...,...,...,...
21317,Zara Extra,Paracetamol + Caffeine,RN Pharmaceuticals
21413,Zerin-XP,Paracetamol + Caffeine,Jayson Pharmaceuticals Ltd.
21414,Zerin,Paracetamol,Jayson Pharmaceuticals Ltd.
21415,Zerin,Paracetamol,Jayson Pharmaceuticals Ltd.


In [62]:
df.head(5).T


,0,1,2,3,4
brand_id,4077,4006,6174,6173,6172
brand_name,A-Cold,A-Cof,A-Clox,A-Clox,A-Clox
type,allopathic,allopathic,allopathic,allopathic,allopathic
dosage_form,Syrup,Syrup,Injection,Injection,Powder for Suspension
generic,Bromhexine Hydrochloride,Dextromethorphan + Pseudoephedrine + Triprolidine,Cloxacillin Sodium,Cloxacillin Sodium,Cloxacillin Sodium
manufacturer,ACME Laboratories Ltd.,ACME Laboratories Ltd.,ACME Laboratories Ltd.,ACME Laboratories Ltd.,ACME Laboratories Ltd.
package_container,100 ml bottle,100 ml bottle,500 mg vial,250 mg vial,100 ml bottle
drug_class,Cough expectorants & mucolytics,Combined cough suppressants,Penicillinase-resistant penicillins,Penicillinase-resistant penicillins,Penicillinase-resistant penicillins
indication,Bronchitis,Non-productive cough,UTI caused by Staphylococcus Aureus,UTI caused by Staphylococcus Aureus,UTI caused by Staphylococcus Aureus
indication_description,Bromhexine is indicated in the treatment of re...,This preparation temporarily relieves these sy...,গ্রাম-পজিটিভ জীবাণু দ্বারা সৃষ্ট সংক্রমণের চিক...,গ্রাম-পজিটিভ জীবাণু দ্বারা সৃষ্ট সংক্রমণের চিক...,গ্রাম-পজিটিভ জীবাণু দ্বারা সৃষ্ট সংক্রমণের চিক...


In [63]:
print("\nMissing values (%):")
print((df.isna().mean() * 100).round(1).sort_values(ascending=False))


Missing values (%):
package_container                      62.5
interaction_description                 9.5
indication                              7.9
precautions_description                 4.6
drug_class                              4.5
therapeutic_class_description           4.4
contraindications_description           3.4
storage_conditions_description          2.9
pharmacology_description                2.8
pregnancy_and_lactation_description     2.3
side_effects_description                0.9
dosage_description                      0.1
type                                    0.0
brand_id                                0.0
brand_name                              0.0
indication_description                  0.0
dosage_form                             0.0
generic                                 0.0
manufacturer                            0.0
dtype: float64
